# 🧪 GIAI ĐOẠN 3 (NHÓM 2): TRANSFORMER ABLATION STUDY & SO SÁNH 3 BỘ TOKENIZER (KAGGLE & COLAB)
**Đề tài 8: Dịch máy Anh – Việt (IWSLT'15 En–Vi) | Môn học: Học sâu**  
**Phụ trách: Sơn (Phần 3 – Nhóm trưởng)**  
---
> **Notebook tự động nhận diện môi trường (chạy được trên cả Kaggle lẫn Google Colab):**
> - **Trên Kaggle:** Bật **Accelerator: GPU P100** (hoặc T4 x2), bật **Internet: ON**, vào **Add-ons → Secrets** tạo & bật secret **`BTL_DL_COLAB`** (W&B API key), sau đó bấm **Save Version → Save & Run All (Commit)** để chạy nền liên tục không lo mất kết nối.
> - **Trên Colab:** Bật **T4 GPU** và bật Colab Secret **`BTL_DL_COLAB`**.
>
> **Đã hoàn thành trước đó (Mốc Base):** `transformer_bpe_base` (`N=4, H=8, PE=True, LS=0.1`) — Greedy `25.76` / Beam(k=4) **`26.71 BLEU`**.
>
> **4 run cốt lõi (`epochs=15`, tắt hoàn toàn spam log `tqdm`, eval `beam_size=4, alpha=0.6`):**
> 1. `transformer_bpe_L2H8` *(Ablation độ sâu: `N=2, H=8`)*
> 2. `transformer_bpe_noPE` *(Ablation: tắt Positional Encoding)*
> 3. `transformer_word_base` *(Tokenizer: Word-level)*
> 4. `transformer_spm_base` *(Tokenizer: SentencePiece)*

### Bước 1: Kiểm tra GPU & Tự nhận diện môi trường (Kaggle / Colab)

In [ ]:
!nvidia-smi

import os

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB = (not IS_KAGGLE) and os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/Machine_Translation_Project"
    STORAGE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"
    ENV_NAME = "Google Colab"
elif IS_KAGGLE:
    PROJECT_DIR = "/kaggle/working/Machine_Translation_Project"
    STORAGE_ROOT = "/kaggle/working/BTL_DL_Translation"
    ENV_NAME = "Kaggle"
else:
    PROJECT_DIR = os.getcwd()
    STORAGE_ROOT = os.path.join(PROJECT_DIR, "local_output")
    ENV_NAME = "Local"

os.makedirs(STORAGE_ROOT, exist_ok=True)
print(f"✓ Môi trường phát hiện: {ENV_NAME}")
print(f"  • PROJECT_DIR  : {PROJECT_DIR}")
print(f"  • STORAGE_ROOT : {STORAGE_ROOT}")

### Bước 2: Clone / Cập nhật mã nguồn nhánh `main` & Cài đặt thư viện

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(
    ["pip", "install", "-q", "sacrebleu", "sentencepiece", "tokenizers", "pyyaml", "wandb", "tabulate"],
    check=True,
)
print(f"✓ Đã sẵn sàng mã nguồn ({BRANCH}) và thư viện tại: {os.getcwd()}")

### Bước 3: Chuẩn bị dữ liệu `data/processed/` & 3 bộ Tokenizer
- **Trên Colab:** Copy thẳng từ Google Drive `BTL_DL_Translation/`.
- **Trên Kaggle:** Tự tải IWSLT'15 qua `download_data.py`, tiền xử lý NFC qua `preprocess.py` và sinh đủ 3 bộ Tokenizer (~30 giây).

In [ ]:
import os
import shutil
import subprocess

os.chdir(PROJECT_DIR)
os.makedirs("data/processed", exist_ok=True)
for sub in ("word", "bpe", "spm"):
    os.makedirs(os.path.join("custom_tokenizers", sub), exist_ok=True)

drive_processed = os.path.join(STORAGE_ROOT, "data", "processed")
drive_tokenizers = os.path.join(STORAGE_ROOT, "custom_tokenizers")

if IS_COLAB and os.path.exists(os.path.join(drive_processed, "train.en")):
    print("[INFO] Colab: Copy data/processed & custom_tokenizers từ Google Drive...")
    shutil.copytree(drive_processed, "data/processed", dirs_exist_ok=True)
    shutil.copytree(drive_tokenizers, "custom_tokenizers", dirs_exist_ok=True)
else:
    if not os.path.exists("data/processed/train.en"):
        print("[INFO] Tải và tiền xử lý IWSLT'15 En-Vi...")
        subprocess.run(["python3", "data/download_data.py"], check=True)
        subprocess.run(["python3", "data/preprocess.py"], check=True)
    if not os.path.exists("custom_tokenizers/bpe/bpe.en.json"):
        subprocess.run(["python3", "custom_tokenizers/train_bpe.py"], check=True)
    if not os.path.exists("custom_tokenizers/spm/spm.en.model"):
        subprocess.run(["python3", "custom_tokenizers/train_spm.py"], check=True)

!ls -lh data/processed/ custom_tokenizers/word/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Auto-login Weights & Biases (`btl-dl-translation-iwslt15`)
Tự động lấy API key từ Secret **`BTL_DL_COLAB`** trên cả **Kaggle Secrets** lẫn **Colab Secrets**.

In [ ]:
import os
import wandb

api_key = None
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("BTL_DL_COLAB")
    except Exception as e:
        print(f"[WARN] Không đọc được Kaggle Secret 'BTL_DL_COLAB': {e}")
elif IS_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get("BTL_DL_COLAB")
    except Exception as e:
        print(f"[WARN] Không đọc được Colab Secret 'BTL_DL_COLAB': {e}")
else:
    api_key = os.environ.get("WANDB_API_KEY")

if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key, relogin=True)
    api = wandb.Api()
    print("✓ Đang đăng nhập W&B:", api.viewer.username, "| Default entity:", api.default_entity)
else:
    print("[WARN] Chưa tìm thấy Secret 'BTL_DL_COLAB' -> các run sẽ chạy không có W&B (để tránh treo khi chạy nền).")

### Bước 5: Định nghĩa hàm `run_experiment()` (Không spam log + Lưu thẳng `STORAGE_ROOT` + Tự resume)

In [ ]:
import json
import os
import re
import subprocess


def _run_clean(cmd: list[str]) -> str:
    """Chạy subprocess, tắt hẳn thanh tqdm và lọc bỏ mọi dòng progress bar gây tràn output."""
    print("\n>>>", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["TQDM_DISABLE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    kept_lines = []
    for line in proc.stdout:
        if "%|" in line or "it/s]" in line or "s/it]" in line:
            continue
        if "Waiting for wandb.init" in line:
            continue
        print(line, end="", flush=True)
        kept_lines.append(line)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {ret}): {' '.join(cmd)}")
    return "".join(kept_lines)


def _parse_bleu(output: str) -> dict:
    res = {}
    m = re.search(r"Điểm BLEU:\s*([0-9.]+)", output)
    if m:
        res["bleu"] = float(m.group(1))
    m_bp = re.search(r"Brevity Penalty:\s*([0-9.]+)", output)
    if m_bp:
        res["bp"] = float(m_bp.group(1))
    m_sig = re.search(r"Signature:\s*(.+)", output)
    if m_sig:
        res["signature"] = m_sig.group(1).strip()
    return res


def run_experiment(
    exp_dir_name: str,
    tokenizer: str = "bpe",
    num_layers: int = 4,
    num_heads: int = 8,
    use_pe: bool = True,
    smoothing: float = 0.1,
    epochs: int = 15,
    batch_size: int = 128,
    skip_if_done: bool = True,
):
    save_dir = os.path.join(STORAGE_ROOT, "checkpoints", exp_dir_name)
    ckpt_path = os.path.join(save_dir, "best_model.pt")
    summary_path = os.path.join(save_dir, "train_summary.json")
    eval_file = os.path.join(save_dir, "eval_result.json")

    if skip_if_done and os.path.exists(eval_file):
        print(f"[SKIP] {exp_dir_name} đã hoàn thành: {eval_file}")
        with open(eval_file, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
        return

    os.makedirs(save_dir, exist_ok=True)

    # 1. Train Transformer (bỏ qua nếu checkpoint + summary đã có sẵn)
    if skip_if_done and os.path.exists(ckpt_path) and os.path.exists(summary_path):
        print(f"[RESUME] Đã có checkpoint & summary của {exp_dir_name} -> bỏ qua train, chạy thẳng eval.py!")
    else:
        _run_clean([
            "python3", "train_transformer.py",
            "--config", "configs/transformer_config.yaml",
            "--tokenizer", tokenizer,
            "--num_layers", str(num_layers),
            "--num_heads", str(num_heads),
            "--use_pe", "true" if use_pe else "false",
            "--smoothing", str(smoothing),
            "--epochs", str(epochs),
            "--batch_size", str(batch_size),
            "--save_dir", save_dir,
        ])

    # 2. Eval bằng Beam Search (k=4, alpha=0.6) trên tst2013 (1,268 câu)
    print(f"\n[INFO] Đang chạy eval.py (beam_size=4) trên tst2013 cho {exp_dir_name} (chờ ~3-5 phút)...", flush=True)
    out_beam4 = _run_clean([
        "python3", "eval.py", "--checkpoint", ckpt_path, "--beam_size", "4", "--alpha", "0.6"
    ])

    eval_result = {
        "exp_dir": exp_dir_name,
        "tokenizer": tokenizer,
        "num_layers": num_layers,
        "num_heads": num_heads,
        "use_pe": use_pe,
        "smoothing": smoothing,
        "beam4": _parse_bleu(out_beam4),
    }
    with open(eval_file, "w", encoding="utf-8") as f:
        json.dump(eval_result, f, ensure_ascii=False, indent=2)

    print(f"\n✓ HOÀN TẤT & ĐÃ LƯU TẠI: {save_dir}")
    print(f"  -> Test BLEU (Beam k=4, alpha=0.6): {eval_result['beam4'].get('bleu')}")

print(f"✓ Đã sẵn sàng hàm run_experiment() trên {ENV_NAME}!")

---
## PHẦN A: 2 BIẾN THỂ ABLATION CỐT LÕI (TRÊN BPE TOKENIZER)

### Run 1/4 — Ablation độ sâu: Giảm số tầng (`N=2, H=8`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_L2H8",
    tokenizer="bpe",
    num_layers=2,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

### Run 2/4 — Ablation Positional Encoding: Tắt PE (`use_pe=False`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_noPE",
    tokenizer="bpe",
    num_layers=4,
    num_heads=8,
    use_pe=False,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

---
## PHẦN B: 2 BỘ TOKENIZER CÒN LẠI TRÊN TRANSFORMER (`Word` & `SentencePiece`)

### Run 3/4 — Tokenizer Word-level (`transformer_word_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_word_base",
    tokenizer="word",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

### Run 4/4 — Tokenizer SentencePiece (`transformer_spm_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_spm_base",
    tokenizer="spm",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=15,
    batch_size=128,
)

---
### *(Tùy chọn nếu còn dư quota GPU)* — 2 biến thể Ablation bổ sung (`H=4` và tắt Label Smoothing)

In [ ]:
# Bỏ dấu # nếu muốn chạy thêm 2 biến thể phụ:
# run_experiment("transformer_bpe_L4H4", tokenizer="bpe", num_layers=4, num_heads=4, use_pe=True, smoothing=0.1, epochs=15)
# run_experiment("transformer_bpe_noLS", tokenizer="bpe", num_layers=4, num_heads=8, use_pe=True, smoothing=0.0, epochs=15)

---
## PHẦN C: TỔNG HỢP BẢNG KẾT QUẢ CSV & JSON (CHÈN VÀO BÁO CÁO ACL)

In [ ]:
import json
import os
import pandas as pd

CKPT_ROOT = os.path.join(STORAGE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(STORAGE_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

EXPERIMENTS = [
    ("transformer_bpe_base",  "Base (BPE, N=4, H=8, PE, LS=0.1)"),
    ("transformer_bpe_L2H8",  "Ablation: N=2 layers (H=8)"),
    ("transformer_bpe_noPE",  "Ablation: No Positional Encoding"),
    ("transformer_bpe_L4H4",  "Ablation: H=4 heads (N=4)"),
    ("transformer_bpe_noLS",  "Ablation: No Label Smoothing (0.0)"),
    ("transformer_word_base", "Tokenizer: Word-level (N=4, H=8)"),
    ("transformer_spm_base",  "Tokenizer: SentencePiece (N=4, H=8)"),
]

# Thông tin mốc Base (đã hoàn thành trên Colab ở notebook 02)
BASE_FALLBACK_ROW = {
    "Config": "Base (BPE, N=4, H=8, PE, LS=0.1)",
    "Run": "transformer-bpe-base",
    "Tokenizer": "bpe",
    "Layers": 4,
    "Heads": 8,
    "PE": True,
    "LS": 0.1,
    "Params": 19676800,
    "Avg_Epoch_s": 165.78,
    "Best_Epoch": 13,
    "Val_Loss": 3.4935,
    "Val_PPL": 12.01,
    "Test_BLEU_Beam4": 26.71,
}

rows = []
for folder, label in EXPERIMENTS:
    s_path = os.path.join(CKPT_ROOT, folder, "train_summary.json")
    e_path = os.path.join(CKPT_ROOT, folder, "eval_result.json")
    if not os.path.exists(s_path):
        if folder == "transformer_bpe_base":
            rows.append(BASE_FALLBACK_ROW)
        continue
    with open(s_path, "r", encoding="utf-8") as f:
        s = json.load(f)
    ev = json.load(open(e_path, "r", encoding="utf-8")) if os.path.exists(e_path) else {}
    m_cfg = s.get("config", {}).get("model", {})
    t_cfg = s.get("config", {}).get("training", {})
    best_ep = min(s.get("history", []), key=lambda r: r["val/loss"], default={})
    bleu4 = ev.get("beam4", {}).get("bleu")
    if bleu4 is None and folder == "transformer_bpe_base":
        bleu4 = 26.71
    rows.append({
        "Config": label,
        "Run": s.get("run_name", folder),
        "Tokenizer": s.get("tokenizer_type"),
        "Layers": m_cfg.get("num_layers"),
        "Heads": m_cfg.get("num_heads"),
        "PE": m_cfg.get("use_pe"),
        "LS": t_cfg.get("label_smoothing"),
        "Params": s.get("trainable_params"),
        "Avg_Epoch_s": s.get("avg_epoch_time_sec"),
        "Best_Epoch": best_ep.get("epoch"),
        "Val_Loss": s.get("best_val_loss"),
        "Val_PPL": best_ep.get("val/perplexity"),
        "Test_BLEU_Beam4": bleu4,
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.csv")
json_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.json")
df.to_csv(csv_out, index=False, encoding="utf-8")
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"✓ Đã xuất bảng tổng hợp ({len(rows)} runs) ra:\n  - {csv_out}\n  - {json_out}\n")
print(df.to_markdown(index=False))